# 05 - Polaris catalog

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

## Open the project

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Dependencies: PostgreSQL and SeaweedFS must be ready. Deployment: `polaris`; Service: `polaris:8181`; health port: 8182. Bootstrap Job: `polaris-bootstrap`. Polaris has no separate PVC: its metadata is in PostgreSQL `polaris` on `postgres-data`, and table files are in S3 on `seaweedfs-data`. Keep the completed bootstrap Job on ordinary restarts.

## Check dependencies

```bash
kubectl rollout status statefulset/seaweedfs --timeout=600s
kubectl rollout status statefulset/postgres --timeout=600s
```



## Apply the bootstrap job

```bash
kubectl apply -f polaris/bootstrap.yaml
```



## Wait for bootstrap

```bash
kubectl wait --for=condition=complete job/polaris-bootstrap --timeout=600s
kubectl logs job/polaris-bootstrap
```

Expect successful initialization. If it fails, inspect the Job logs; do not continue.

## Apply the catalog server

```bash
kubectl apply -f polaris/polaris.yaml
```



## Wait for Polaris

```bash
kubectl rollout status deployment/polaris --timeout=600s
kubectl get deployment polaris
kubectl get service polaris
```



## Forward the catalog API

```bash
if [[ -f .state/polaris-forward.pid ]]; then
  FORWARD_PID=$(cat .state/polaris-forward.pid)
  if [[ "$(ps -p "$FORWARD_PID" -o args= 2>/dev/null || true)" == 'kubectl port-forward --address=127.0.0.1 service/polaris 18181:8181 18182:8182' ]]; then kill "$FORWARD_PID"; fi
fi
kubectl port-forward --address=127.0.0.1 service/polaris 18181:8181 18182:8182 > .state/polaris-forward.log 2>&1 &
echo $! > .state/polaris-forward.pid
```



## Check API readiness

```bash
for attempt in $(seq 1 60); do
  if curl -fsS http://127.0.0.1:18182/q/health/ready > .state/polaris-health.json; then break; fi
  sleep 1
done
curl -fsS http://127.0.0.1:18182/q/health/ready
```



## Obtain a token

```bash
POLARIS_URL=http://127.0.0.1:18181
CLIENT_ID=$(kubectl get secret lab-credentials -o jsonpath='{.data.polaris-client-id}' | base64 -d)
CLIENT_SECRET=$(kubectl get secret lab-credentials -o jsonpath='{.data.polaris-client-secret}' | base64 -d)
TOKEN=$(curl -fsS "$POLARIS_URL/api/catalog/v1/oauth/tokens" -H 'Polaris-Realm: POLARIS' --data-urlencode grant_type=client_credentials --data-urlencode "client_id=$CLIENT_ID" --data-urlencode "client_secret=$CLIENT_SECRET" --data-urlencode scope=PRINCIPAL_ROLE:ALL | python3 -c 'import json,sys; print(json.load(sys.stdin)["access_token"])')
api() { curl -fsS -H "Authorization: Bearer $TOKEN" -H 'Polaris-Realm: POLARIS' -H 'Content-Type: application/json' "$@"; }
```

Keep the token in this terminal. Obtain a new token after a Polaris restart.

## Prepare test_catalog

```bash
python3 - <<'PY'
import json
from pathlib import Path
body = json.loads(Path('polaris/catalog.json').read_text())
body['catalog']['name'] = 'test_catalog'
body['catalog']['properties']['default-base-location'] = 's3://test-bucket/iceberg'
body['catalog']['storageConfigInfo']['allowedLocations'] = ['s3://test-bucket/iceberg']
Path('.state/test-catalog.json').write_text(json.dumps(body, indent=2))
PY
```

The S3 endpoint is Kubernetes service DNS. It must not be localhost inside Polaris.

## Create or inspect test_catalog

```bash
CATALOG_STATUS=$(curl -sS -o .state/test-catalog-response.json -w '%{http_code}' -H "Authorization: Bearer $TOKEN" -H 'Polaris-Realm: POLARIS' "$POLARIS_URL/api/management/v1/catalogs/test_catalog")
case "$CATALOG_STATUS" in
  404) api -X POST --data-binary @.state/test-catalog.json "$POLARIS_URL/api/management/v1/catalogs" ;;
  200) cat .state/test-catalog-response.json ;;
  *) cat .state/test-catalog-response.json; false ;;
esac
api "$POLARIS_URL/api/management/v1/catalogs/test_catalog"
```



## Prepare the reusable exercise catalog

```bash
CATALOG_STATUS=$(curl -sS -o .state/training-catalog-response.json -w '%{http_code}' -H "Authorization: Bearer $TOKEN" -H 'Polaris-Realm: POLARIS' "$POLARIS_URL/api/management/v1/catalogs/training")
case "$CATALOG_STATUS" in
  404) api -X POST --data-binary @polaris/catalog.json "$POLARIS_URL/api/management/v1/catalogs" ;;
  200) cat .state/training-catalog-response.json ;;
  *) cat .state/training-catalog-response.json; false ;;
esac
```

Later notebooks reuse catalog `training` and bucket `warehouse`. The extra test catalog stays separate.

## Create test_namespace

```bash
NAMESPACE_STATUS=$(curl -sS -o .state/test-namespace-response.json -w '%{http_code}' -H "Authorization: Bearer $TOKEN" -H 'Polaris-Realm: POLARIS' "$POLARIS_URL/api/catalog/v1/test_catalog/namespaces/test_namespace")
case "$NAMESPACE_STATUS" in
  404) api -X POST --data '{"namespace":["test_namespace"]}' "$POLARIS_URL/api/catalog/v1/test_catalog/namespaces" ;;
  200) cat .state/test-namespace-response.json ;;
  *) cat .state/test-namespace-response.json; false ;;
esac
```



## Create an Iceberg test_table

```bash
TABLE_STATUS=$(curl -sS -o .state/test-table-response.json -w '%{http_code}' -H "Authorization: Bearer $TOKEN" -H 'Polaris-Realm: POLARIS' "$POLARIS_URL/api/catalog/v1/test_catalog/namespaces/test_namespace/tables/test_table")
case "$TABLE_STATUS" in
  404) api -X POST --data '{"name":"test_table","schema":{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"id","required":true,"type":"int"}]}}' "$POLARIS_URL/api/catalog/v1/test_catalog/namespaces/test_namespace/tables" ;;
  200) cat .state/test-table-response.json ;;
  *) cat .state/test-table-response.json; false ;;
esac
api "$POLARIS_URL/api/catalog/v1/test_catalog/namespaces/test_namespace/tables/test_table" > .state/iceberg-table.json
python3 -c 'import json; t=json.load(open(".state/iceberg-table.json")); assert t["metadata-location"].startswith("s3://test-bucket/iceberg/"); print(t["metadata-location"])'
```

This creates Iceberg schema and metadata files, not data rows. Spark/Trino row writes follow in later sessions.

## Close the API forward

```bash
kill "$(cat .state/polaris-forward.pid)"
rm .state/polaris-forward.pid
```



## Check first-start logs and metrics

```bash
python3 validations/check-observability.py --components polaris
```

Monitoring was started in notebook 02. This checks the actual first-start daemon logs and CPU/memory samples without restarting the service to generate logs.


## Stop this service

```bash
kubectl scale deployment/polaris --replicas=0
kubectl wait --for=delete pod -l app=polaris --timeout=180s
```

This retains its Service and storage. Run after finishing the examples.

## Resume this service

```bash
kubectl scale deployment/polaris --replicas=1
kubectl rollout status deployment/polaris --timeout=600s
```

